# Cuaderno 05 — Análisis de series de tiempo en franja diurna e imputación

**Proyecto de Grado — Maestría en Ciencia de Datos**
**Pontificia Universidad Javeriana Cali — Grupo 8**

---

## Para qué existe este cuaderno

Reemplaza el análisis descriptivo del cuaderno 03 y responde a once observaciones de la
dirección del trabajo de grado. La idea que las une es una sola:

> **La mitad de la serie son ceros de la noche.** Toda estadística calculada sobre la serie
> completa describe esa mitad muerta y no el fenómeno que se quiere modelar. La mediana de
> cero, el coeficiente de variación del 187 %, las colas raras del gráfico cuantil a cuantil y
> la sensación de que la imputación «no conserva la tendencia» son todas manifestaciones del
> mismo problema.

La decisión adoptada es **trabajar el análisis descriptivo sobre la franja diurna**, de 6:00 a
18:55, y reportar la serie completa solo como contexto.

## Qué observación atiende cada sección

| Sección | Punto de la reunión |
|---|---|
| 3. Descriptivo en franja diurna | 12, 16, 17 |
| 4. Diagnóstico propio de series de tiempo (ACF, PACF, tendencia, estacionariedad) | 4, 9 |
| 5. Visualización multiescala | 11 |
| 6. Efecto de la imputación, visto de cerca | 10 |
| 7. Comparación por nivel de agregación | 25 |
| 8. Validación de la imputación por ocultamiento | 25, 26 |
| 9. Conservación de la tendencia | 13, 14, 15, 26 |
| 10. Índice de claridad corregido | 18 |

## Antes de ejecutar
Requiere `dataset_final_interpolacion.csv`, `dataset_final_media_clim.csv` y
`dataset_final_kalman.csv` del cuaderno 02, y el archivo original de la estación.

## 0. Importaciones y estilo

In [ ]:
!pip install pandas numpy matplotlib statsmodels scipy pykalman --quiet

import os, subprocess, warnings
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from scipy import stats as sps

from statsmodels.tsa.stattools import acf, pacf, adfuller, kpss
from statsmodels.tsa.seasonal import STL

warnings.filterwarnings('ignore')
SEED = 42
np.random.seed(SEED)

# Franja diurna del documento: 6:00 a 18:55 inclusive.
HORA_INI_DIURNO = 6
HORA_FIN_DIURNO = 18

# El estilo de las figuras (paleta y tamanos de letra) se define en la celda
# siguiente, la celda de estilo, para que sea identico en los 11 cuadernos.


print('Librerias importadas:', datetime.now().strftime('%Y-%m-%d %H:%M:%S'))


### Estilo unico de figuras

Esta celda fija la **paleta de colores** y los **tamanos de letra** de todas las figuras del cuaderno. Es la misma celda en los once cuadernos, de modo que cualquier figura del trabajo de grado sale con el mismo aspecto.


In [ ]:
# @@ESTILO_TG@@  (marca unica: no editar esta linea)
# =============================================================================
#  ESTILO UNICO DE FIGURAS DEL TRABAJO DE GRADO
#  Esta celda define la paleta y la tipografia de TODAS las figuras.
#  Es identica en los 11 cuadernos: cambiarla aqui cambia solo este cuaderno,
#  por eso si se modifica hay que modificarla en todos.
#
#  Paleta categorica anclada en el azul. Validada con el verificador de
#  accesibilidad de color (separacion OKLab bajo protanopia y deuteranopia,
#  piso de croma, banda de luminosidad y contraste contra fondo blanco):
#  todos los pares de colores pasan, no solo los contiguos.
# =============================================================================
import os
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from cycler import cycler
try:
    import seaborn as sns
except Exception:
    sns = None

# --- 1. Paleta categorica (orden FIJO: la ranura 1 es siempre la serie principal)
C_AZUL    = '#0C60A3'   # ranura 1 - serie principal / observado / T1 / V1
C_AMBAR   = '#D4771A'   # ranura 2 - segunda serie / modelo / T2 / V2
C_CELESTE = '#269AE6'   # ranura 3 - tercera serie / T3 / V3
C_TEJA    = '#963E35'   # ranura 4 - cuarta serie
C_VERDEAZ = '#0D8F73'   # ranura 5 - quinta serie
PALETA_TG = [C_AZUL, C_AMBAR, C_CELESTE, C_TEJA, C_VERDEAZ]

# --- 2. Tonos de apoyo de la rampa azul (rellenos, bandas, sombreados)
C_AZUL_CLARO  = '#81B4EA'
C_AZUL_MEDIO  = '#5A9BDC'
C_AZUL_OSCURO = '#0A3765'

# --- 3. Tinta neutra: texto, ejes, lineas de referencia. NUNCA es una serie.
C_TEXTO      = '#1A1A1A'
C_GRIS       = '#6E6E6E'
C_GRIS_CLARO = '#C9C9C9'

# --- 4. Mapas de color continuos
#   'tg_azul' : secuencial de un solo tono, claro -> oscuro (magnitudes)
#   'tg_div'  : divergente azul <-> teja con gris neutro en el centro
#               (mantiene la lectura de 'RdBu_r': negativo azul, positivo calido)
_SECUENCIAL = ['#F4F9FF', '#D8E8FA', '#B4D2F4', '#81B4EA', '#5A9BDC',
               '#3581C9', '#1667AC', '#0B4E89', '#0A3765']
_DIVERGENTE = ['#0A3765', '#1667AC', '#5A9BDC', '#AFCEEC', '#F2F1EF',
               '#EFC9AC', '#DC9A5A', '#B4642C', '#7E3A2C']
CMAP_SEC = LinearSegmentedColormap.from_list('tg_azul', _SECUENCIAL)
CMAP_DIV = LinearSegmentedColormap.from_list('tg_div',  _DIVERGENTE)

def _registrar_cmap(cm_obj):
    """Registra el mapa por nombre para poder usar cmap='tg_azul'."""
    try:
        mpl.colormaps.register(cm_obj, force=True)          # matplotlib >= 3.6
    except Exception:
        try:
            mpl.cm.register_cmap(cmap=cm_obj)               # matplotlib antiguo
        except Exception:
            pass

for _c in (CMAP_SEC, CMAP_DIV):
    _registrar_cmap(_c)

def escala_azul(n, ini=0.375, fin=1.0):
    """n colores tomados de la rampa azul. Para variables ORDENADAS (anios,
    meses, horizontes): el orden se lee en el tono, de claro a oscuro."""
    if n <= 1:
        return [C_AZUL]
    paso = (fin - ini) / (n - 1)
    return [CMAP_SEC(ini + i * paso) for i in range(n)]

# --- 6. Eje X de fechas legible ----------------------------------------------
import matplotlib.dates as mdates

def eje_fechas(ax):
    """Ajusta el eje X de fechas al rango real de la serie: anios cuando abarca
    anios, meses cuando abarca meses. Evita que se repita la misma etiqueta
    ('2023 2023 2023 ...') cuando el periodo es corto."""
    loc = mdates.AutoDateLocator(minticks=4, maxticks=9)
    ax.xaxis.set_major_locator(loc)
    ax.xaxis.set_major_formatter(mdates.ConciseDateFormatter(loc))
    return ax

# --- 7. Trazo por serie: si dos curvas se superponen, el trazo las separa -----
ESTILOS_TG = ['-', '--', ':', '-.', (0, (5, 1, 1, 1))]
TRAZO = {'T1': '-', 'T2': '--', 'T3': ':',
         'V1': '-', 'V2': '--', 'V3': ':'}

# --- 5. Tipografia: tamanos grandes para que la figura se lea en el PDF impreso
TAM_BASE = 13          # texto general dentro de la figura
TAM_EJE  = 14          # etiquetas de los ejes X e Y
TAM_TICK = 12          # numeros de los ejes
TAM_TIT  = 14.5        # titulo del panel
TAM_LEY  = 12          # leyenda

plt.rcParams.update({
    # --- color
    'axes.prop_cycle'   : cycler(color=PALETA_TG),
    'image.cmap'        : 'tg_azul',
    'figure.facecolor'  : 'white',
    'axes.facecolor'    : 'white',
    'savefig.facecolor' : 'white',
    # --- tipografia (lo que pidio la revision: ejes y etiquetas legibles)
    'font.family'       : 'sans-serif',
    'font.size'         : TAM_BASE,
    'axes.titlesize'    : TAM_TIT,
    'axes.labelsize'    : TAM_EJE,
    'axes.titleweight'  : 'bold',
    'axes.labelweight'  : 'normal',
    'xtick.labelsize'   : TAM_TICK,
    'ytick.labelsize'   : TAM_TICK,
    'legend.fontsize'   : TAM_LEY,
    'legend.title_fontsize': TAM_LEY,
    'figure.titlesize'  : TAM_TIT + 1.5,
    # --- trazos y ejes discretos
    'lines.linewidth'   : 1.8,
    'lines.markersize'  : 5.5,
    'axes.linewidth'    : 0.9,
    'axes.edgecolor'    : '#8A8A8A',
    'axes.labelcolor'   : C_TEXTO,
    'text.color'        : C_TEXTO,
    'xtick.color'       : '#8A8A8A',
    'ytick.color'       : '#8A8A8A',
    'xtick.labelcolor'  : C_TEXTO,
    'ytick.labelcolor'  : C_TEXTO,
    'xtick.major.width' : 0.9,
    'ytick.major.width' : 0.9,
    'grid.color'        : C_GRIS_CLARO,
    'grid.linewidth'    : 0.7,
    'grid.alpha'        : 0.55,
    'legend.frameon'    : True,
    'legend.framealpha' : 0.92,
    'legend.edgecolor'  : '#C9C9C9',
    # --- exportacion para Overleaf: el texto viaja como texto dentro del PDF
    'figure.dpi'        : 110,
    'savefig.dpi'       : 300,
    'savefig.bbox'      : 'tight',
    'pdf.fonttype'      : 42,
    'ps.fonttype'       : 42,
    'svg.fonttype'      : 'path',
})

if sns is not None:
    sns.set_palette(PALETA_TG)

print('Estilo de figuras aplicado.')
print(f'  Paleta categorica: {", ".join(PALETA_TG)}')
print(f'  Mapas continuos:   tg_azul (secuencial), tg_div (divergente)')
print(f'  Tamanos: ejes {TAM_EJE} pt, numeros {TAM_TICK} pt, leyenda {TAM_LEY} pt')

# --- Nombres que usa este cuaderno, atados a la paleta unica ---------------
C_ORIG = C_TEXTO    # serie original observada (tinta neutra, no es una serie mas)
C_T1   = C_AZUL     # interpolacion lineal
C_T2   = C_AMBAR    # media climatologica
C_T3   = C_CELESTE  # filtro de Kalman
COL    = {'T1': C_T1, 'T2': C_T2, 'T3': C_T3}
ETQ    = {'T1': 'T1 interpolacion lineal',
          'T2': 'T2 media climatologica',
          'T3': 'T3 filtro de Kalman'}


## 1. Rutas y carga

In [ ]:
# @@RUTAS_TG@@  (marca unica: no editar esta linea)
# =============================================================================
#  DONDE VIVE EL PROYECTO
#
#  En el Drive hay mas de una copia del proyecto: la del semestre anterior y la
#  actual. Antes cada cuaderno buscaba un archivo de datos por todo el Drive y
#  se quedaba con el PRIMERO que aparecia, que no siempre era el de la carpeta
#  correcta; por eso 'figuras', 'figuras_PDF' y 'figuras_SVG' se crearon en la
#  carpeta vieja.
#
#  Ahora la carpeta del proyecto se decide UNA sola vez y de forma explicita:
#  es la carpeta cuyo nombre esta en NOMBRE_CARPETA y cuya ruta contiene
#  CARPETA_PROYECTO. TODO lo que el cuaderno escriba va ahi.
#
#  Si algun dia mueves el proyecto (por ejemplo a 'Proyecto Aplicado IV'),
#  cambias UNA linea: CARPETA_PROYECTO. Y la cambias en los once cuadernos.
# =============================================================================
import os
import subprocess

CARPETA_PROYECTO = 'Proyecto Aplicado III'      # <-- la carpeta actual del semestre
NOMBRE_CARPETA   = 'Codigos Trabajo de Grado'   # <-- nombre de la carpeta de codigo
RAIZ_DRIVE       = '/content/drive/MyDrive'

# Rutas que nunca se usan como carpeta del proyecto, aunque coincidan de nombre.
EXCLUIR_RUTAS = ['/anterior/', '/.Trash', '/.shortcut-targets-by-id/',
                 '/copia de ', '/copy of ']


def _montar_drive():
    try:
        from google.colab import drive
    except Exception:
        return
    if not os.path.isdir(RAIZ_DRIVE):
        drive.mount('/content/drive')
    else:
        print('Drive ya montado.')


def _find(patron, tipo='f', maxdepth=None, raiz=RAIZ_DRIVE, timeout=300):
    cmd = ['find', raiz]
    if maxdepth is not None:
        cmd += ['-maxdepth', str(maxdepth)]
    cmd += ['-type', tipo, '-name', patron]
    try:
        res = subprocess.run(cmd, capture_output=True, text=True, timeout=timeout)
    except Exception:
        return []
    return [r for r in res.stdout.strip().split('\n') if r]


def _excluida(ruta):
    r = ruta.lower()
    return any(x.lower() in r for x in EXCLUIR_RUTAS)


def localizar_proyecto():
    """Devuelve la carpeta del proyecto. Falla con un mensaje claro si no la
    encuentra o si encuentra mas de una: nunca elige a ciegas."""
    _montar_drive()

    candidatas = _find(NOMBRE_CARPETA, tipo='d', maxdepth=8)
    if not candidatas:
        candidatas = _find(NOMBRE_CARPETA, tipo='d')
    # Segunda pista: cualquier carpeta que contenga los cuadernos de la cadena.
    for nb in _find('0*_*REVISADO.ipynb', tipo='f', maxdepth=9):
        d = os.path.dirname(nb)
        if d not in candidatas:
            candidatas.append(d)

    validas   = [c for c in candidatas if CARPETA_PROYECTO in c and not _excluida(c)]
    descartes = [c for c in candidatas if c not in validas]

    if len(validas) == 1:
        print(f'Carpeta del proyecto: {validas[0]}')
        if descartes:
            print(f'  ({len(descartes)} carpeta(s) descartada(s) por no estar en '
                  f'"{CARPETA_PROYECTO}")')
        return validas[0]

    print('\n' + '=' * 78)
    print('NO SE PUDO DECIDIR LA CARPETA DEL PROYECTO')
    print('=' * 78)
    print(f'  Se busca:  una carpeta "{NOMBRE_CARPETA}" cuya ruta contenga '
          f'"{CARPETA_PROYECTO}"')
    print(f'  Encontradas: {len(candidatas)}')
    for c in candidatas:
        marca = 'SIRVE     ' if c in validas else 'descartada'
        print(f'    [{marca}] {c}')
    print()
    if not validas:
        print('  Que hacer: comprueba el nombre exacto de la carpeta en Drive y ajusta')
        print('  arriba CARPETA_PROYECTO o NOMBRE_CARPETA. Si moviste el proyecto,')
        print('  recuerda cambiarlo en los once cuadernos.')
    else:
        print('  Hay varias carpetas que sirven. Escribe arriba una CARPETA_PROYECTO')
        print('  mas especifica, por ejemplo la ruta completa desde MyDrive.')
    print('=' * 78 + '\n')
    raise FileNotFoundError(
        f'No hay exactamente una carpeta "{NOMBRE_CARPETA}" bajo "{CARPETA_PROYECTO}".')


# --- La carpeta del proyecto y todo lo que cuelga de ella --------------------
RUTA_BASE     = localizar_proyecto()
RUTA_DATOS    = os.path.join(RUTA_BASE, 'datos')
RUTA_ARRAYS   = os.path.join(RUTA_DATOS, 'arrays')
RUTA_MODELOS  = os.path.join(RUTA_BASE, 'modelos')
RUTA_RES      = os.path.join(RUTA_BASE, 'resultados')
RUTA_FIGS     = os.path.join(RUTA_BASE, 'figuras')       # PNG
RUTA_FIGS_SVG = os.path.join(RUTA_BASE, 'figuras_SVG')   # SVG
RUTA_FIGS_PDF = os.path.join(RUTA_BASE, 'figuras_PDF')   # PDF, el de Overleaf

for _d in (RUTA_DATOS, RUTA_ARRAYS, RUTA_MODELOS, RUTA_RES,
           RUTA_FIGS, RUTA_FIGS_SVG, RUTA_FIGS_PDF):
    os.makedirs(_d, exist_ok=True)


def archivo_de_datos(nombre, obligatorio=True, pista=''):
    """Localiza un archivo de ENTRADA.

    Primero lo busca dentro de la carpeta del proyecto. Solo si no esta lo busca
    en el resto del Drive, y en ese caso avisa en grande: significa que ese
    archivo se quedo en la carpeta vieja y conviene copiarlo.
    Lo que el cuaderno ESCRIBE nunca sale de la carpeta del proyecto.
    """
    for sub in ('datos', os.path.join('datos', 'arrays'), '', 'resultados', 'modelos'):
        cand = os.path.join(RUTA_BASE, sub, nombre) if sub else os.path.join(RUTA_BASE, nombre)
        if os.path.exists(cand):
            return cand

    fuera = _find(nombre, tipo='f')
    if fuera:
        fuera.sort(key=lambda r: os.path.getmtime(r), reverse=True)
        elegido = fuera[0]
        print('\n' + '!' * 78)
        print(f'AVISO: "{nombre}" no esta en la carpeta del {CARPETA_PROYECTO}.')
        print(f'  Se leera de: {elegido}')
        print(f'  Copialo a  : {RUTA_DATOS}')
        print('  Mientras siga en dos sitios, corres el riesgo de leer una version vieja.')
        print('!' * 78 + '\n')
        return elegido

    if obligatorio:
        raise FileNotFoundError(
            f'No se encontro "{nombre}" ni en la carpeta del proyecto ni en el resto '
            f'del Drive.\n  Carpeta del proyecto: {RUTA_BASE}\n  {pista}')
    return None


print(f'  datos      : {RUTA_DATOS}')
print(f'  figuras    : {RUTA_FIGS} (+ _SVG y _PDF)')
print(f'  modelos    : {RUTA_MODELOS}')
print(f'  resultados : {RUTA_RES}')

# --- Entradas propias de este cuaderno ---------------------------------------
archivo_de_datos('dataset_final_interpolacion.csv',
                 pista='Lo genera el cuaderno 02.')
RUTA_ORIG = archivo_de_datos('Datos_2013_2026_completo.csv',
                             pista='Es el CSV de la estacion CESMAG, sin procesar.')
print('  RUTA_ORIG :', RUTA_ORIG)


### Figuras en PNG, SVG y PDF

**Añadido en la revisión.** Todas las figuras de este cuaderno se guardan ahora en los tres
formatos, con el mismo nombre y en tres carpetas paralelas: `figuras/` para el PNG,
`figuras_SVG/` para el vectorial editable y `figuras_PDF/` para el que se incluye en Overleaf.

En LaTeX se incluye sin extensión, y así el compilador elige el formato adecuado:

```latex
\includegraphics[width=\linewidth]{figuras_PDF/fig_01b_distribucion}
```

In [ ]:
# --- Guardado de figuras en los tres formatos ----------------------------
#
# El documento se compila en Overleaf, que incluye las figuras en PDF, pero
# conviene tener tambien el SVG (editable) y el PNG (vista rapida y para
# presentaciones). Antes cada cuaderno guardaba en un formato distinto: unos en
# SVG, otros en PDF, otros en PNG. Ahora todos usan esta funcion y producen los
# tres, con el mismo nombre, en tres carpetas paralelas:
#
#   figuras/       nombre.png   300 ppp
#   figuras_SVG/   nombre.svg   vectorial editable
#   figuras_PDF/   nombre.pdf   el que se incluye en LaTeX
#
# En LaTeX basta con \includegraphics{figuras_PDF/nombre} sin extension.
import os
import matplotlib.pyplot as plt

FORMATOS_FIGURA = (
    ('RUTA_FIGS',     'png', {'dpi': 300}),
    ('RUTA_FIGS_SVG', 'svg', {}),
    ('RUTA_FIGS_PDF', 'pdf', {}),
)


def guardar_figura(fig, nombre, verbose=True):
    """Guarda la figura en PNG, SVG y PDF. 'nombre' va SIN extension."""
    if fig is None:
        fig = plt.gcf()
    base = os.path.splitext(os.path.basename(str(nombre)))[0]
    rutas = []
    for var, ext, extra in FORMATOS_FIGURA:
        carpeta = globals().get(var)
        if carpeta is None:
            continue
        os.makedirs(carpeta, exist_ok=True)
        ruta = os.path.join(carpeta, f'{base}.{ext}')
        fig.savefig(ruta, bbox_inches='tight', **extra)
        rutas.append(ruta)
    if verbose:
        print(f'  Figura guardada: {base}.png, .svg y .pdf')
    return rutas


print('Funcion guardar_figura lista: cada figura se guarda en PNG, SVG y PDF.')
for _v in ('RUTA_FIGS', 'RUTA_FIGS_SVG', 'RUTA_FIGS_PDF'):
    print(f'  {_v:<14} {globals().get(_v)}')

In [ ]:
# Serie original, sin imputar
with open(RUTA_ORIG, 'r', encoding='utf-8', errors='replace') as f:
    lin = [f.readline().strip() for _ in range(3)]
SEP = ';' if lin[0].count(';') > lin[0].count(',') else ','
DEC = ',' if (SEP == ';' and ',' in lin[1]) else '.'

df_orig = pd.read_csv(RUTA_ORIG, sep=SEP, decimal=DEC,
                      parse_dates=['DateTime'], index_col='DateTime')
df_orig.columns = ['RS']
df_orig.index.name = 'timestamp'
df_orig['RS'] = pd.to_numeric(df_orig['RS'], errors='coerce')
df_orig.sort_index(inplace=True)

# Correcciones fisicas de la Etapa 1, para que la comparacion sea justa
h = df_orig.index.hour
mask_diurno_o = (h >= HORA_INI_DIURNO) & (h <= HORA_FIN_DIURNO)
df_orig.loc[df_orig['RS'] < 0, 'RS'] = 0.0
df_orig.loc[(df_orig['RS'] > 0) & ~mask_diurno_o, 'RS'] = np.nan

# Series imputadas
DATASETS = {}
for k, nom in [('T1', 'interpolacion'), ('T2', 'media_clim'), ('T3', 'kalman')]:
    DATASETS[k] = pd.read_csv(os.path.join(RUTA_DATOS, f'dataset_final_{nom}.csv'),
                              index_col='timestamp', parse_dates=True)

idx = DATASETS['T1'].index
hh  = idx.hour
MASK_DIURNO = (hh >= HORA_INI_DIURNO) & (hh <= HORA_FIN_DIURNO)

print(f'Original : {len(df_orig):,} registros, {df_orig["RS"].isna().sum():,} faltantes')
for k in DATASETS:
    print(f'{k}       : {len(DATASETS[k]):,} registros, '
          f'{DATASETS[k]["RS"].isna().sum()} faltantes')
print(f'Franja diurna: {MASK_DIURNO.sum():,} de {len(idx):,} registros '
      f'({MASK_DIURNO.mean()*100:.2f} %)')


## 2. Cuánto pesa la noche

Antes de nada, el número que justifica toda la reestructuración. Se cuantifica qué fracción
de la serie son ceros y de dónde vienen.

In [ ]:
# Serie diurna del original CONSERVANDO los faltantes: se usa en todo el cuaderno
obs_diurna = df_orig.loc[mask_diurno_o, 'RS']

rs_o_todo  = df_orig['RS'].dropna()
rs_o_diur  = obs_diurna.dropna()
rs_o_pos   = rs_o_diur[rs_o_diur > 0]

print('DE DONDE SALEN LOS CEROS')
print('=' * 66)
print(f'  Observaciones validas, serie completa : {len(rs_o_todo):>10,}')
print(f'    de ellas iguales a cero             : {int((rs_o_todo==0).sum()):>10,}  '
      f'({(rs_o_todo==0).mean()*100:.2f} %)')
print(f'  Observaciones validas, franja diurna  : {len(rs_o_diur):>10,}')
print(f'    de ellas iguales a cero             : {int((rs_o_diur==0).sum()):>10,}  '
      f'({(rs_o_diur==0).mean()*100:.2f} %)')
print(f'  Observaciones diurnas positivas       : {len(rs_o_pos):>10,}')
print()
print('  Mediana serie completa  :', f'{rs_o_todo.median():.2f} W/m2')
print('  Mediana franja diurna   :', f'{rs_o_diur.median():.2f} W/m2')
print('  Mediana diurnos > 0     :', f'{rs_o_pos.median():.2f} W/m2')
print()
print('  Lectura: la mediana de cero de la serie completa no describe la irradiancia,')
print('  describe la duracion de la noche. Los ceros que quedan dentro de la franja')
print('  diurna se concentran en los bordes, en torno al amanecer y al ocaso.')

# Reparto horario de los ceros dentro de la franja diurna
ceros_hora = (rs_o_diur[rs_o_diur == 0].groupby(rs_o_diur[rs_o_diur == 0].index.hour)
              .size())
print()
print('  Ceros dentro de la franja diurna, por hora:')
print(ceros_hora.to_string())


## 3. Estadísticas descriptivas en franja diurna

Reemplaza la Tabla 6.3 del documento. Tres cambios respecto de la versión actual:

1. **Se calcula sobre la franja diurna**, no sobre la serie completa.
2. **Desaparecen la asimetría y la curtosis.** Son momentos de la distribución marginal; en
   una serie de tiempo lo que interesa es la estructura temporal, y en una serie con una masa
   de puntos en cero esos momentos describen la masa en cero y no la irradiancia.
3. **Se añade el rango intercuartílico y el coeficiente de dispersión por cuartiles**, que son
   medidas de variabilidad robustas y que no dependen de que la media sea un valor típico.

Se reportan dos versiones: la franja diurna completa y la franja diurna restringida a valores
positivos. La comparación entre ambas es la que responde a la pregunta de si conviene analizar
solo las horas con sol.

In [ ]:
def descriptivas(s, nombre):
    v = pd.Series(s).dropna()
    q1, q2, q3 = v.quantile([0.25, 0.5, 0.75])
    return pd.Series({
        'N'          : len(v),
        'Ceros'      : int((v == 0).sum()),
        'Minimo'     : v.min(),
        'Maximo'     : v.max(),
        'Media'      : v.mean(),
        'Mediana'    : q2,
        'Desv_est'   : v.std(),
        'CV_pct'     : v.std() / v.mean() * 100 if v.mean() else np.nan,
        'RIC'        : q3 - q1,
        'CDC'        : (q3 - q1) / (q3 + q1) if (q3 + q1) else np.nan,
        'P10'        : v.quantile(0.10),
        'P25'        : q1,
        'P75'        : q3,
        'P90'        : v.quantile(0.90),
        'P95'        : v.quantile(0.95),
        'P99'        : v.quantile(0.99),
    }, name=nombre)


# --- Version A: franja diurna completa -------------------------------------
colA = [descriptivas(rs_o_diur, 'Original')]
for k in ['T1', 'T2', 'T3']:
    colA.append(descriptivas(DATASETS[k].loc[MASK_DIURNO, 'RS'], k))
tabla_diurna = pd.concat(colA, axis=1)

# --- Version B: franja diurna con valores positivos ------------------------
colB = [descriptivas(rs_o_pos, 'Original')]
for k in ['T1', 'T2', 'T3']:
    s = DATASETS[k].loc[MASK_DIURNO, 'RS']
    colB.append(descriptivas(s[s > 0], k))
tabla_diurna_pos = pd.concat(colB, axis=1)

# --- Contexto: serie completa, solo para una fila de la tabla del documento -
colC = [descriptivas(rs_o_todo, 'Original')]
for k in ['T1', 'T2', 'T3']:
    colC.append(descriptivas(DATASETS[k]['RS'], k))
tabla_completa = pd.concat(colC, axis=1)

print('A) FRANJA DIURNA (6:00 a 18:55) — W/m2')
print('=' * 72); print(tabla_diurna.round(2).to_string())
print()
print('B) FRANJA DIURNA, SOLO VALORES POSITIVOS — W/m2')
print('=' * 72); print(tabla_diurna_pos.round(2).to_string())
print()
print('C) SERIE COMPLETA, solo como contexto — W/m2')
print('=' * 72); print(tabla_completa.round(2).to_string())

for nom, t in [('diurna', tabla_diurna), ('diurna_pos', tabla_diurna_pos),
               ('completa', tabla_completa)]:
    t.to_csv(os.path.join(RUTA_DATOS, f'tabla_09_descriptivas_{nom}.csv'))

print()
print('EL COEFICIENTE DE VARIACION, EXPLICADO')
print('-' * 66)
for etq, t in [('Serie completa', tabla_completa), ('Franja diurna', tabla_diurna),
               ('Diurna positiva', tabla_diurna_pos)]:
    print(f'  {etq:<18} CV original = {t.loc["CV_pct","Original"]:7.2f} %   '
          f'CV tras T1 = {t.loc["CV_pct","T1"]:7.2f} %')
print()
print('  El coeficiente de variacion es la desviacion dividida por la media. Al imputar,')
print('  la media sube porque se agregan valores diurnos positivos que antes no se')
print('  contaban, mientras la desviacion apenas cambia. El cociente baja por aritmetica,')
print('  no porque la serie se haya vuelto mas estable. Restringido a la franja diurna el')
print('  indicador vuelve a ser interpretable.')


## 4. Diagnóstico propio de series de tiempo

Esto es lo que faltaba por completo en el documento. Un análisis descriptivo de series de
tiempo no se agota en medias y percentiles: hay que mirar la estructura temporal.

Se hacen cuatro cosas:

1. **Descomposición en climatología y anomalía.** La irradiancia tiene una parte determinista,
   impuesta por la posición del Sol, y una parte estocástica, causada por las nubes. Se separa
   restando el promedio histórico de cada combinación de hora y mes. La parte interesante para
   el modelado es la anomalía.
2. **Función de autocorrelación y autocorrelación parcial.** La primera dice cuánto se parece
   la serie a sí misma desplazada; la segunda, cuánto de ese parecido es directo y no heredado
   de retardos intermedios.
3. **Tendencia.** Pendiente de una recta ajustada a la serie diaria y prueba de Mann-Kendall,
   que no supone linealidad ni normalidad.
4. **Estacionariedad.** Pruebas de Dickey-Fuller aumentada y de KPSS, que son complementarias:
   la primera tiene por hipótesis nula que la serie no es estacionaria y la segunda, que sí lo
   es. Conviene reportar las dos.

Una advertencia técnica sobre la autocorrelación: si se elimina la noche, la serie queda con
huecos y los retardos dejan de significar lo mismo. Por eso la autocorrelación a cinco minutos
se calcula sobre la serie **continua**, y el análisis diurno se hace sobre la serie **diaria**,
que sí es continua día a día.

In [ ]:
SERIE = DATASETS['T1']['RS']            # serie de trabajo: la imputada seleccionada

# --- 4.1 Climatologia y anomalia -------------------------------------------
aux = pd.DataFrame({'v': SERIE.values}, index=SERIE.index)
aux['hora'] = aux.index.hour
aux['min']  = aux.index.minute
aux['mes']  = aux.index.month
clim = aux.groupby(['mes', 'hora', 'min'])['v'].transform('mean')
aux['clim'] = clim
aux['anom'] = aux['v'] - aux['clim']

var_total = aux['v'].var()
var_clim  = aux['clim'].var()
var_anom  = aux['anom'].var()
print('DESCOMPOSICION DE LA VARIANZA')
print('=' * 66)
print(f'  Varianza total                       : {var_total:12,.1f}')
print(f'  Varianza de la climatologia          : {var_clim:12,.1f}  '
      f'({var_clim/var_total*100:5.2f} %)')
print(f'  Varianza de la anomalia              : {var_anom:12,.1f}  '
      f'({var_anom/var_total*100:5.2f} %)')
print()
print('  La primera es la parte que la geometria solar explica por si sola.')
print('  La segunda es la que el modelo tiene que aprender: la nubosidad.')

# --- 4.2 Series a distintas escalas -----------------------------------------
serie_5min   = SERIE
serie_diaria = SERIE.loc[MASK_DIURNO].resample('D').mean().dropna()
serie_mensual= SERIE.loc[MASK_DIURNO].resample('MS').mean().dropna()
anom_5min    = aux['anom']

print()
print(f'  Serie a 5 minutos : {len(serie_5min):,} puntos')
print(f'  Serie diaria      : {len(serie_diaria):,} puntos (promedio de la franja diurna)')
print(f'  Serie mensual     : {len(serie_mensual):,} puntos')


In [ ]:
# --- 4.3 ACF y PACF ---------------------------------------------------------
N_LAGS_5MIN = 576      # dos dias a cinco minutos
N_LAGS_DIA  = 400      # algo mas de un anio

acf_5   = acf(serie_5min.values,   nlags=N_LAGS_5MIN, fft=True)
acf_an  = acf(anom_5min.values,    nlags=N_LAGS_5MIN, fft=True)
acf_d   = acf(serie_diaria.values, nlags=N_LAGS_DIA,  fft=True)
pacf_d  = pacf(serie_diaria.values, nlags=40, method='ywm')
pacf_h  = pacf(SERIE.resample('h').mean().dropna().values, nlags=48, method='ywm')

fig, axes = plt.subplots(2, 2, figsize=(14, 8))

ax = axes[0, 0]
ejeh = np.arange(len(acf_5)) * 5 / 60
ax.plot(ejeh, acf_5, color=C_ORIG, lw=1.4, label='Serie observada')
ax.plot(ejeh, acf_an, color=C_T2, lw=1.4, label='Anomalia respecto de la climatologia')
ax.axhline(0, color=C_GRIS, lw=0.9)
ax.set_xlabel('Retardo (horas)')
ax.set_ylabel('Autocorrelacion')
ax.legend(frameon=False)
ax.grid(True, alpha=0.3)

ax = axes[0, 1]
ax.plot(np.arange(len(acf_d)), acf_d, color=C_T1, lw=1.4)
ax.axhline(0, color=C_GRIS, lw=0.9)
ax.axvline(365, color=C_T2, lw=1.5, ls='--', label='Retardo de un anio')
ax.set_xlabel('Retardo (dias)')
ax.set_ylabel('Autocorrelacion')
ax.legend(frameon=False)
ax.grid(True, alpha=0.3)

ax = axes[1, 0]
lim = 1.96 / np.sqrt(len(SERIE.resample('h').mean().dropna()))
ax.vlines(np.arange(len(pacf_h)), 0, pacf_h, color=C_T1, lw=2)
ax.axhline(0, color=C_GRIS, lw=0.9)
ax.fill_between([0, len(pacf_h)-1], -lim, lim, color=C_AZUL_CLARO, alpha=0.35,
                label='Banda de no significancia al 95 %')
ax.set_xlabel('Retardo (horas)')
ax.set_ylabel('Autocorrelacion parcial')
ax.legend(frameon=False)
ax.grid(True, alpha=0.3)

ax = axes[1, 1]
lim_d = 1.96 / np.sqrt(len(serie_diaria))
ax.vlines(np.arange(len(pacf_d)), 0, pacf_d, color=C_T3, lw=2)
ax.axhline(0, color=C_GRIS, lw=0.9)
ax.fill_between([0, len(pacf_d)-1], -lim_d, lim_d, color=C_AZUL_CLARO, alpha=0.35,
                label='Banda de no significancia al 95 %')
ax.set_xlabel('Retardo (dias)')
ax.set_ylabel('Autocorrelacion parcial')
ax.legend(frameon=False)
ax.grid(True, alpha=0.3)

fig.tight_layout()
guardar_figura(fig, 'fig_09a_acf_pacf')
plt.show()

print('LECTURA')
print('-' * 66)
print(f'  ACF a 5 min, retardo 1   : {acf_5[1]:.4f}   (serie observada)')
print(f'  ACF a 5 min, retardo 1   : {acf_an[1]:.4f}   (anomalia)')
print(f'  ACF a 5 min, retardo 288 : {acf_5[288]:.4f}   (un dia exacto)')
print(f'  ACF diaria, retardo 365  : {acf_d[365]:.4f}   (un anio exacto)')
print(f'  PACF horaria, retardo 1  : {pacf_h[1]:.4f}')
print(f'  PACF horaria, retardo 2  : {pacf_h[2]:.4f}')
print()
print('  El pico de la ACF en el retardo de un dia es la firma del ciclo diurno.')
print('  La caida rapida de la ACF de la anomalia mide la memoria real del proceso')
print('  de nubosidad, que es lo que acota el horizonte util de pronostico.')

pd.DataFrame({'retardo_min': np.arange(len(acf_5))*5,
              'acf_observada': acf_5, 'acf_anomalia': acf_an}
             ).to_csv(os.path.join(RUTA_DATOS, 'tabla_09_acf_5min.csv'), index=False)


In [ ]:
# --- 4.4 Tendencia y estacionariedad ---------------------------------------
t = np.arange(len(serie_diaria))
pend, inter, r_val, p_val, err = sps.linregress(t, serie_diaria.values)
pend_anual = pend * 365.25
tau, p_mk = sps.kendalltau(t, serie_diaria.values)

print('TENDENCIA DE LA SERIE DIARIA DIURNA')
print('=' * 66)
print(f'  Pendiente             : {pend:.6f} W/m2 por dia')
print(f'  Equivalente anual     : {pend_anual:+.3f} W/m2 por anio')
print(f'  Valor p de la recta   : {p_val:.4g}')
print(f'  Tau de Mann-Kendall   : {tau:+.4f}   (valor p = {p_mk:.4g})')
print(f'  Cambio en el periodo  : {pend*len(serie_diaria):+.2f} W/m2 '
      f'sobre una media de {serie_diaria.mean():.2f} W/m2 '
      f'({pend*len(serie_diaria)/serie_diaria.mean()*100:+.2f} %)')

sub = serie_diaria.values
adf_st, adf_p = adfuller(sub, autolag='AIC')[:2]
kpss_st, kpss_p = kpss(sub, regression='c', nlags='auto')[:2]
print()
print('ESTACIONARIEDAD DE LA SERIE DIARIA DIURNA')
print('=' * 66)
print(f'  Dickey-Fuller aumentada : estadistico = {adf_st:.4f}, valor p = {adf_p:.4g}')
print('    hipotesis nula: la serie NO es estacionaria')
print(f'  KPSS                    : estadistico = {kpss_st:.4f}, valor p = {kpss_p:.4g}')
print('    hipotesis nula: la serie SI es estacionaria')

# Descomposicion estacional de la serie diaria
stl = STL(serie_diaria.asfreq('D').interpolate(), period=365, robust=True).fit()

fig, axes = plt.subplots(4, 1, figsize=(13, 10), sharex=True)
for ax, comp, etq in zip(
        axes,
        [serie_diaria, stl.trend, stl.seasonal, stl.resid],
        ['Serie diaria observada', 'Componente de tendencia',
         'Componente estacional anual', 'Componente irregular']):
    ax.plot(comp.index, comp.values, color=C_T1, lw=0.9)
    ax.set_ylabel('W/m$^2$')
    ax.grid(True, alpha=0.3)
    ax.text(0.006, 0.90, etq, transform=ax.transAxes, fontsize=12, va='top',
            color=C_TEXTO)
axes[-1].set_xlabel('Fecha')
eje_fechas(axes[-1])
fig.tight_layout()
guardar_figura(fig, 'fig_09b_descomposicion')
plt.show()

print()
print(f'  Amplitud de la componente estacional : '
      f'{stl.seasonal.max()-stl.seasonal.min():.2f} W/m2')
print(f'  Rango de la componente de tendencia  : '
      f'{stl.trend.max()-stl.trend.min():.2f} W/m2')


## 5. Visualización multiescala

Responde a la pregunta de si conviene presentar la serie completa o tramos con zoom.
La respuesta es **las dos cosas, porque cada una muestra algo distinto**:

- La serie completa a resolución diaria muestra estacionalidad y tendencia. A resolución de
  cinco minutos no muestra nada: 1,3 millones de puntos sobre quince centímetros de papel son
  unos noventa mil puntos por centímetro, una mancha.
- Un mes a resolución horaria muestra la alternancia de días despejados y cubiertos.
- Tres días a resolución nativa muestran la variabilidad de minutos que es la que el modelo
  debe capturar.

Se eligen además tres días representativos por su índice de claridad: uno despejado, uno
cubierto y uno con realce por nubes.

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(13, 10))

# Panel 1: periodo completo, resolucion diaria
ax = axes[0]
ax.plot(serie_diaria.index, serie_diaria.values, color=C_T1, lw=0.5, alpha=0.55,
        label='Promedio diario de la franja diurna')
ma90 = serie_diaria.rolling(90, center=True, min_periods=30).mean()
ax.plot(ma90.index, ma90.values, color=C_ORIG, lw=2.0,
        label='Media movil de 90 dias')
ax.set_ylabel('Irradiancia (W/m$^2$)')
ax.set_xlabel('Fecha')
ax.legend(frameon=False, loc='upper left', ncol=2)
ax.grid(True, alpha=0.3)
eje_fechas(ax)

# Panel 2: un mes, resolucion horaria
# PARCHE DE LA REVISION: el mes se elige solo, en vez de estar escrito a mano.
# Se toma el mes con mas observaciones validas de la franja diurna, que es el
# mas representativo y, sobre todo, uno que siempre existe en la serie.
_cuenta = SERIE.dropna().resample('MS').size()
MES_ZOOM = _cuenta.idxmax().strftime('%Y-%m')
sub_mes = SERIE.loc[MES_ZOOM].resample('h').mean()
ax = axes[1]
ax.plot(sub_mes.index, sub_mes.values, color=C_T1, lw=1.0)
ax.set_ylabel('Irradiancia (W/m$^2$)')
ax.set_xlabel(f'Dia de {MES_ZOOM}')
ax.grid(True, alpha=0.3)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%d'))

# Panel 3: tres dias representativos, resolucion nativa
kt_d = (DATASETS['T1'].loc[MASK_DIURNO, 'KT'].resample('D').mean().dropna()
        if 'KT' in DATASETS['T1'].columns else None)
if kt_d is not None:
    dia_desp = kt_d.idxmax().date()
    dia_cub  = kt_d.idxmin().date()
else:
    dia_desp = serie_diaria.idxmax().date()
    dia_cub  = serie_diaria.idxmin().date()
dia_realce = SERIE.idxmax().date()

ax = axes[2]
for d, etq, c in [(dia_desp, 'Dia despejado', C_T2),
                  (dia_cub,  'Dia cubierto',  C_T1),
                  (dia_realce, 'Dia con realce por nubes', C_T3)]:
    s = SERIE.loc[str(d)]
    ax.plot(s.index.hour + s.index.minute / 60, s.values, lw=1.4, color=c,
            label=f'{etq} ({d})')
ax.set_xlabel('Hora del dia')
ax.set_ylabel('Irradiancia (W/m$^2$)')
ax.set_xlim(5, 19)
ax.legend(frameon=False)
ax.grid(True, alpha=0.3)

fig.tight_layout()
guardar_figura(fig, 'fig_09c_multiescala')
plt.show()
print('Guardado: fig_09c_multiescala.pdf')
print(f'Dias seleccionados: despejado {dia_desp}, cubierto {dia_cub}, realce {dia_realce}')


## 6. El efecto de la imputación, visto de cerca

El cuaderno 02 genera esta figura y nunca llegó al documento. Es la que responde de forma más
directa a la pregunta por el efecto de cada técnica: sobre una ventana con faltantes reales,
se ven los valores observados, la serie reconstruida y los puntos que cada técnica inventó.

In [ ]:
# Ventanas con faltantes: una racha corta y una racha larga
falt = df_orig['RS'].isna() & mask_diurno_o
bloques = (falt != falt.shift()).cumsum()
rachas = (falt[falt].groupby(bloques[falt])
          .agg(ini=lambda s: s.index.min(), fin=lambda s: s.index.max(), n='size'))
rachas['horas'] = rachas['n'] * 5 / 60

corta = rachas[rachas['n'] <= 24].sort_values('n', ascending=False).iloc[0]
larga = rachas[rachas['n'] > 24].sort_values('n').iloc[len(rachas[rachas['n']>24])//2]

print(f'Racha corta elegida: {corta["ini"]} a {corta["fin"]}  ({corta["horas"]:.1f} h)')
print(f'Racha larga elegida: {larga["ini"]} a {larga["fin"]}  ({larga["horas"]:.1f} h)')

fig, axes = plt.subplots(2, 1, figsize=(13, 8))
for ax, r, etq in [(axes[0], corta, 'Racha corta, dentro del limite de dos horas'),
                   (axes[1], larga, 'Racha larga, por encima del limite de dos horas')]:
    ini = r['ini'].normalize() - pd.Timedelta(days=1)
    fin = r['fin'].normalize() + pd.Timedelta(days=2)
    ven = slice(ini, fin)
    orig_v = df_orig.loc[ven, 'RS']
    nanm   = orig_v.isna()

    for k in ['T1', 'T2', 'T3']:
        s = DATASETS[k].loc[ven, 'RS']
        ax.plot(s.index, s.values, color=COL[k], lw=1.2, alpha=0.85, label=ETQ[k])
    ax.plot(orig_v.dropna().index, orig_v.dropna().values, '.', color=C_ORIG, ms=2.5,
            label='Valores observados', zorder=5)
    ax.axvspan(r['ini'], r['fin'], color=C_GRIS_CLARO, alpha=0.45, zorder=0,
               label='Tramo imputado')
    ax.set_ylabel('Irradiancia (W/m$^2$)')
    ax.legend(frameon=False, ncol=3, loc='upper right', fontsize=11)
    ax.grid(True, alpha=0.3)
    ax.text(0.006, 0.94, etq, transform=ax.transAxes, fontsize=12, va='top',
            color=C_TEXTO)
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%d/%m %H:%M'))

axes[-1].set_xlabel('Fecha y hora')
fig.tight_layout()
guardar_figura(fig, 'fig_09d_imputacion_de_cerca')
plt.show()
print('Guardado: fig_09d_imputacion_de_cerca.pdf')


## 7. Comparación por nivel de agregación

La observación de la dirección sobre el artículo de precipitación: los autores no compararon
las técnicas sobre la serie diaria, que es ruidosa, sino sobre la mensual. Aquí se hace a los
tres niveles para ver hasta dónde llega ese efecto.

Hay que tener cuidado con qué se compara. A resolución nativa la serie imputada **coincide
exactamente** con la original allí donde la original tiene dato, porque la imputación solo
rellena huecos: comparar ahí da error cero y no informa de nada. Las dos comparaciones que sí
informan son:

1. **Entre técnicas, en los puntos imputados.** Es donde las tres difieren, y mide cuánta
   libertad hay en la elección de método.
2. **Contra el agregado del original, por nivel.** El promedio del original se calcula sobre lo
   observado; el de la serie imputada, sobre todo. La distancia entre ambos mide cuánto
   desplaza la imputación el nivel de cada período, y es la que debería reducirse al agregar.

In [ ]:
# --- (1) Diferencia entre tecnicas, solo en los puntos que tuvieron que inventar ---
idx_imputados = obs_diurna.index[obs_diurna.isna()]
print('DIFERENCIA ENTRE TECNICAS EN LOS PUNTOS IMPUTADOS (resolucion nativa)')
print('=' * 78)
print(f'  Puntos imputados en franja diurna: {len(idx_imputados):,}')
print()
for a, b in [('T1', 'T2'), ('T1', 'T3'), ('T2', 'T3')]:
    d = (DATASETS[a].loc[idx_imputados, 'RS'] - DATASETS[b].loc[idx_imputados, 'RS']).abs()
    print(f'  |{a} - {b}|   media = {d.mean():7.3f}   maxima = {d.max():8.2f} W/m2')
print()
print('  Diferencias pequenias aqui significan que las tres tecnicas reconstruyen casi lo')
print('  mismo, no que las tres acierten. Quien decide el acierto es la validacion por')
print('  ocultamiento de la seccion siguiente.')

# --- (2) Desplazamiento del nivel frente al original, por nivel de agregacion ---
niveles = {}
for nom, regla in [('Diario', 'D'), ('Semanal', 'W'), ('Mensual', 'MS')]:
    fila = {}
    cnt  = obs_diurna.resample(regla).count()
    tot  = obs_diurna.resample(regla).size()
    base = obs_diurna.resample(regla).mean().where(cnt / tot >= 0.6)
    for k in ['T1', 'T2', 'T3']:
        s = DATASETS[k].loc[MASK_DIURNO, 'RS'].resample(regla).mean()
        comun = base.dropna().index.intersection(s.index)
        d = (s.loc[comun] - base.loc[comun]).dropna()
        fila[f'{k}_MAE']   = d.abs().mean()
        fila[f'{k}_sesgo'] = d.mean()
        fila[f'{k}_corr']  = s.loc[comun].corr(base.loc[comun])
    fila['n_periodos'] = int(base.notna().sum())
    niveles[nom] = fila

tabla_niveles = pd.DataFrame(niveles).T
print()
print('DESPLAZAMIENTO FRENTE AL PROMEDIO OBSERVADO, POR NIVEL DE AGREGACION')
print('=' * 84)
print('(solo periodos con al menos 60 % de cobertura en el original)')
print()
print(tabla_niveles.round(3).to_string())
tabla_niveles.to_csv(os.path.join(RUTA_DATOS, 'tabla_09_por_agregacion.csv'))

print()
print('  Si la correlacion sube al agregar, se confirma lo que la direccion senalo a')
print('  proposito del articulo de precipitacion: a resolucion fina la comparacion esta')
print('  dominada por el ruido, y la concordancia entre serie imputada y observada solo')
print('  se aprecia con claridad en escalas agregadas.')


## 8. Validación de la imputación por ocultamiento

Este es el experimento que el documento no tiene y que resuelve la discusión de raíz.

Hasta ahora las tres técnicas se comparan por su parecido con la distribución observada, lo
que solo dice que no distorsionan el conjunto. No dice **cuál acierta más**, porque los valores
verdaderos de los huecos no se conocen.

El experimento consiste en **esconder datos que sí se tienen**, imitando la estructura real de
los huecos, imputarlos y comparar contra la verdad. Es la única forma de decir con evidencia
cuál técnica es mejor.

Estructura de los huecos artificiales: se toman longitudes de la distribución real de rachas,
y se colocan sobre días completamente observados.

In [ ]:
from pykalman import KalmanFilter

# Longitudes reales de racha, para imitar la estructura
long_reales = rachas['n'].values
print(f'Rachas reales: {len(long_reales)}, mediana {np.median(long_reales):.0f} intervalos, '
      f'maxima {long_reales.max():.0f}')

# Dias completamente observados en franja diurna
obs_por_dia = obs_diurna.groupby(obs_diurna.index.date).apply(lambda s: s.notna().mean())
dias_completos = [d for d, p in obs_por_dia.items() if p == 1.0]
print(f'Dias con franja diurna completa: {len(dias_completos):,}')

rng = np.random.default_rng(SEED)
N_HUECOS = 400
dias_sel = rng.choice(dias_completos, size=min(N_HUECOS, len(dias_completos)), replace=False)
longs    = rng.choice(long_reales, size=len(dias_sel), replace=True)

serie_exp = obs_diurna.copy()
verdad_idx = []
for d, L in zip(dias_sel, longs):
    dia = serie_exp.loc[str(d)]
    L = int(min(L, len(dia)))
    if L < 1:
        continue
    ini = rng.integers(0, max(1, len(dia) - L + 1))
    idxs = dia.index[ini:ini + L]
    verdad_idx.extend(idxs)

verdad_idx = pd.DatetimeIndex(sorted(set(verdad_idx)))
verdad = serie_exp.loc[verdad_idx].copy()
serie_exp.loc[verdad_idx] = np.nan
print(f'Puntos ocultados: {len(verdad_idx):,} '
      f'({len(verdad_idx)/len(obs_diurna)*100:.2f} % de la franja diurna)')


In [ ]:
# Reimplementacion local de las tres tecnicas, identica a la del cuaderno 02
LIM = 24   # 2 horas

base_clim = obs_diurna.dropna()
dic_clim = (base_clim.groupby([base_clim.index.hour, base_clim.index.month])
            .mean().to_dict())

def imp_clim(s):
    out = s.copy()
    m = out.isna()
    out.loc[m] = [dic_clim.get((ts.hour, ts.month), 0.0) for ts in out.index[m]]
    return out.clip(0, 1600)

def imp_T1(s):
    out = s.interpolate(method='time', limit=LIM, limit_direction='both')
    return imp_clim(out).clip(0, 1600)

def imp_T2(s):
    return imp_clim(s)

def imp_T3(s):
    out = s.copy()
    for d, seg in out.groupby(out.index.date):
        if seg.isna().sum() == 0 or seg.notna().sum() < 10:
            continue
        run = seg.isna().astype(int)
        blo = (run != run.shift()).cumsum()
        maxrun = run[run == 1].groupby(blo[run == 1]).sum().max() if run.any() else 0
        if maxrun > LIM:
            continue
        v = seg.values.astype(float)
        mu, va = np.nanmean(v), max(np.nanvar(v), 1.0)
        kf = KalmanFilter(transition_matrices=[[1]], observation_matrices=[[1]],
                          transition_covariance=[[va*0.1]],
                          observation_covariance=[[va*0.5]],
                          initial_state_mean=[mu], initial_state_covariance=[[va]],
                          em_vars=['transition_covariance','observation_covariance'])
        obs = np.ma.masked_invalid(v.reshape(-1,1))
        try:
            kf = kf.em(obs, n_iter=5)
            est, _ = kf.smooth(obs)
            nuevo = seg.copy()
            nuevo[seg.isna()] = est[:,0][seg.isna().values]
            out.loc[seg.index] = nuevo.values
        except Exception:
            pass
    return imp_clim(out).clip(0, 1600)

print('Aplicando las tres tecnicas sobre la serie con huecos artificiales...')
res_imp = {'T1': imp_T1(serie_exp), 'T2': imp_T2(serie_exp), 'T3': imp_T3(serie_exp)}
print('Listo.')


In [ ]:
# Metricas frente a la verdad oculta, globales y por clase de racha
def clase_racha(idx_serie):
    m = idx_serie.isna()
    b = (m != m.shift()).cumsum()
    L = m[m].groupby(b[m]).transform('size')
    return L

L_por_punto = clase_racha(serie_exp).reindex(verdad_idx)

filas = []
for k, s in res_imp.items():
    pred = s.loc[verdad_idx]
    e = pred - verdad
    for etq, sel in [('Todas',      pd.Series(True, index=verdad_idx)),
                     ('Racha <= 2h', L_por_punto <= LIM),
                     ('Racha > 2h',  L_por_punto >  LIM)]:
        sel = sel.reindex(verdad_idx).fillna(False)
        if sel.sum() < 10:
            continue
        filas.append({
            'tecnica': ETQ[k], 'clase': etq, 'n': int(sel.sum()),
            'MAE'  : e[sel].abs().mean(),
            'RMSE' : np.sqrt((e[sel]**2).mean()),
            'sesgo': e[sel].mean(),
            'corr' : pred[sel].corr(verdad[sel]),
        })

val_imp = pd.DataFrame(filas)
print('VALIDACION POR OCULTAMIENTO — error frente al valor verdadero (W/m2)')
print('=' * 84)
print(val_imp.round(3).to_string(index=False))
val_imp.to_csv(os.path.join(RUTA_DATOS, 'tabla_09_validacion_imputacion.csv'), index=False)

# --- Registro de la tecnica ganadora, para que 04 y 10 la lean ------------
# Sin este archivo habria que cambiar a mano el nombre del conjunto en varios
# cuadernos, y basta olvidarse en uno para que el documento describa una
# tecnica y el modelo use otra.
SUFIJO = {'T1 interpolacion lineal': 'interpolacion',
          'T2 media climatologica' : 'media_clim',
          'T3 filtro de Kalman'    : 'kalman'}

mejor = val_imp[val_imp['clase'] == 'Todas'].sort_values('RMSE').iloc[0]
_suf = SUFIJO.get(mejor['tecnica'])
if _suf is None:
    for _k, _v in SUFIJO.items():
        if _k.split()[0] in str(mejor['tecnica']):
            _suf = _v
            break
if _suf is None:
    _suf = 'interpolacion'
with open(os.path.join(RUTA_DATOS, 'tecnica_imputacion.txt'), 'w',
          encoding='utf-8') as f:
    f.write(_suf)
print()
print(f'Registrado en tecnica_imputacion.txt: {_suf}')
print('Los cuadernos 04 y 10 leeran ese archivo; no hay que cambiar nada a mano.')
print()
print(f'  Tecnica con menor error global: {mejor["tecnica"]} '
      f'(RMSE = {mejor["RMSE"]:.2f} W/m2)')
print('  Ahora la seleccion de tecnica descansa en evidencia de acierto y no solo en')
print('  parecido distribucional.')

# Figura de dispersion imputado contra verdadero
fig, axes = plt.subplots(1, 3, figsize=(15, 4.6), sharex=True, sharey=True)
for ax, k in zip(axes, ['T1', 'T2', 'T3']):
    pred = res_imp[k].loc[verdad_idx]
    ax.scatter(verdad.values, pred.values, s=3, alpha=0.15, color=COL[k])
    lim = float(max(verdad.max(), pred.max()))
    ax.plot([0, lim], [0, lim], color=C_ORIG, lw=1.2, ls='--',
            label='Linea de igualdad')
    ax.set_xlabel('Irradiancia verdadera (W/m$^2$)')
    ax.legend(frameon=False)
    ax.grid(True, alpha=0.3)
    ax.text(0.03, 0.94, ETQ[k], transform=ax.transAxes, fontsize=12, va='top')
axes[0].set_ylabel('Irradiancia imputada (W/m$^2$)')
fig.tight_layout()
guardar_figura(fig, 'fig_09e_validacion_imputacion')
plt.show()


## 9. Conservación de la tendencia

La preocupación de fondo de la dirección: en la figura de perfiles la línea punteada del
original queda por debajo de las imputadas y parece que la imputación no conserva la forma.

**El desplazamiento tiene una causa aritmética y no metodológica.** El perfil mensual del
original se calcula sobre las observaciones disponibles de cada mes; en los meses con muchos
faltantes diurnos, ese promedio se calcula sobre una muestra en la que sobran las horas de
poca irradiancia. Al imputar, esas casillas se llenan con valores diurnos y el promedio sube.
La distancia entre las dos curvas es, mes a mes, proporcional a la fracción de datos que
faltaba.

Se demuestra de tres maneras:

1. Superponiendo la **fracción de datos faltantes por mes** al desplazamiento observado.
2. Recalculando el perfil del original **solo con meses de cobertura alta**, donde el
   desplazamiento debe desaparecer.
3. Midiendo la **correlación entre las formas** de ambos perfiles, que es lo que de verdad
   significa conservar la tendencia.

In [ ]:
# Perfil mensual sobre franja diurna
perf_orig = obs_diurna.groupby(obs_diurna.index.month).mean()
cob_mes   = obs_diurna.groupby(obs_diurna.index.month).apply(lambda s: s.notna().mean())
perf_imp  = {k: DATASETS[k].loc[MASK_DIURNO, 'RS'].groupby(
                 DATASETS[k].loc[MASK_DIURNO].index.month).mean() for k in ['T1','T2','T3']}

# Version con cobertura alta: solo pares anio-mes con al menos 90 % de datos.
# La mascara se construye sobre el indice COMPLETO para que sirva igual en el
# original y en los tres conjuntos imputados.
assert (df_orig.index == DATASETS['T1'].index).all(), \
    'El indice del original y el de los conjuntos imputados deben coincidir.'

cob_ma = obs_diurna.groupby([obs_diurna.index.year,
                             obs_diurna.index.month]).apply(lambda s: s.notna().mean())
buenos = set(cob_ma[cob_ma >= 0.90].index)

idx_full = DATASETS['T1'].index
sel_full = np.array([(a, m) in buenos for a, m in zip(idx_full.year, idx_full.month)])
sel_diur = sel_full & MASK_DIURNO
print(f'Pares anio-mes con cobertura de al menos 90 %: {len(buenos)}')
print(f'Registros diurnos que aportan al perfil de cobertura alta: {sel_diur.sum():,}')

MI = pd.Index(range(1, 13))
perf_orig_alta = (df_orig.loc[sel_diur, 'RS']
                  .groupby(df_orig.index[sel_diur].month).mean().reindex(MI))
perf_imp_alta  = {k: (DATASETS[k].loc[sel_diur, 'RS']
                      .groupby(DATASETS[k].index[sel_diur].month).mean().reindex(MI))
                  for k in ['T1', 'T2', 'T3']}

perf_orig = perf_orig.reindex(MI)
perf_imp  = {k: v.reindex(MI) for k, v in perf_imp.items()}
cob_mes   = cob_mes.reindex(MI)

MESES = ['Ene','Feb','Mar','Abr','May','Jun','Jul','Ago','Sep','Oct','Nov','Dic']

fig, axes = plt.subplots(1, 3, figsize=(16, 4.8))

ax = axes[0]
ax.plot(perf_orig.index, perf_orig.values, color=C_ORIG, lw=2.4, ls='--',
        label='Original observado')
for k in ['T1','T2','T3']:
    ax.plot(perf_imp[k].index, perf_imp[k].values, color=COL[k], lw=1.8, label=ETQ[k])
ax.set_xticks(range(1,13)); ax.set_xticklabels(MESES, rotation=45)
ax.set_xlabel('Mes'); ax.set_ylabel('Irradiancia diurna promedio (W/m$^2$)')
ax.legend(frameon=False, fontsize=11); ax.grid(True, alpha=0.3)
ax.text(0.03,0.96,'Todos los meses', transform=ax.transAxes, va='top', fontsize=12)

ax = axes[1]
ax.plot(perf_orig_alta.index, perf_orig_alta.values, color=C_ORIG, lw=2.4, ls='--',
        label='Original observado')
for k in ['T1','T2','T3']:
    ax.plot(perf_imp_alta[k].index, perf_imp_alta[k].values, color=COL[k], lw=1.8,
            label=ETQ[k])
ax.set_xticks(range(1,13)); ax.set_xticklabels(MESES, rotation=45)
ax.set_xlabel('Mes'); ax.set_ylabel('Irradiancia diurna promedio (W/m$^2$)')
ax.legend(frameon=False, fontsize=11); ax.grid(True, alpha=0.3)
ax.text(0.03,0.96,'Solo meses con cobertura de al menos 90 %',
        transform=ax.transAxes, va='top', fontsize=12)

ax = axes[2]
desfase = (perf_imp['T1'] - perf_orig)
ax.bar(desfase.index, (1-cob_mes).values*100, color=C_GRIS_CLARO,
       label='Porcentaje de datos faltantes')
ax2 = ax.twinx()
ax2.plot(desfase.index, desfase.values, color=C_T1, lw=2.2, marker='o', ms=5,
         label='Desplazamiento del promedio')
ax.set_xticks(range(1,13)); ax.set_xticklabels(MESES, rotation=45)
ax.set_xlabel('Mes'); ax.set_ylabel('Datos faltantes (%)')
ax2.set_ylabel('Desplazamiento (W/m$^2$)')
ax.grid(True, alpha=0.3, axis='y')
h1,l1 = ax.get_legend_handles_labels(); h2,l2 = ax2.get_legend_handles_labels()
ax.legend(h1+h2, l1+l2, frameon=False, fontsize=11, loc='upper left')

fig.tight_layout()
guardar_figura(fig, 'fig_09f_conservacion_tendencia')
plt.show()

def corr_sin_nan(a, b):
    d = pd.concat([pd.Series(np.asarray(a, dtype=float)),
                   pd.Series(np.asarray(b, dtype=float))], axis=1).dropna()
    return np.nan if len(d) < 3 else d.iloc[:, 0].corr(d.iloc[:, 1])

print('CONSERVACION DE LA FORMA DEL PERFIL MENSUAL')
print('=' * 70)
_r_falt = corr_sin_nan(1-cob_mes.values, desfase.values)
_desp   = float(np.nanmean(np.abs(desfase.values)))
print(f'  Desplazamiento medio del perfil mensual      : {_desp:.2f} W/m2')
print(f'  Correlacion entre faltantes y desplazamiento : {_r_falt:.4f}')
if _desp < 1.0:
    print('  El desplazamiento ya es practicamente nulo, asi que esta correlacion no')
    print('  significa nada: no queda efecto que explicar. Se conserva solo como control.')
elif _r_falt > 0.6:
    print('  Correlacion alta con un desplazamiento no nulo: el desplazamiento sigue al')
    print('  faltante, es decir es un artefacto de cuantos datos hay y no de la imputacion.')
else:
    print('  Correlacion baja con un desplazamiento no nulo: el desplazamiento NO se explica')
    print('  por el faltante y conviene mirarlo de cerca antes de seguir.')
print()
for k in ['T1','T2','T3']:
    r_todos = corr_sin_nan(perf_orig.values, perf_imp[k].values)
    r_altos = corr_sin_nan(perf_orig_alta.values, perf_imp_alta[k].values)
    d_todos = (perf_imp[k]-perf_orig).mean()
    d_altos = (perf_imp_alta[k]-perf_orig_alta).mean()
    print(f'  {ETQ[k]:<26} forma r = {r_todos:.4f} (todos) / {r_altos:.4f} (cobertura alta)')
    print(f'  {"":<26} nivel   = {d_todos:+7.2f} / {d_altos:+7.2f} W/m2')
print()
print('  La correlacion entre formas mide conservacion de tendencia. La diferencia de')
print('  nivel mide el efecto de contabilizar datos que antes faltaban, y no es un')
print('  defecto de la imputacion.')


## 9b. Perfiles con mediana y banda intercuartílica

Responde a tres observaciones de golpe: si el promedio es la mejor estadística para mostrar el
perfil (puntos 13 y 14), qué sentido tiene el perfil por hora del día (punto 7) y por qué un
gráfico de barras con un promedio en el eje vertical no es adecuado (punto 6).

**El promedio no es la mejor estadística aquí.** La distribución de la irradiancia en una hora
dada es asimétrica y con mucha dispersión: días despejados y días cubiertos conviven en la
misma casilla. El promedio los mezcla en un número que no describe bien a ninguno, y es además
sensible a que la imputación agregue masa. La mediana con una banda entre el percentil 25 y el
75 dice dos cosas en lugar de una: dónde está el día típico y cuánto varía.

**El gráfico de barras se sustituye por líneas.** Una barra codifica una cantidad mediante un
área que arranca en cero, y se usa para comparar categorías. La hora del día y el mes no son
categorías: son posiciones en un ciclo continuo. La línea es la representación correcta, y
además permite superponer las cuatro series sin que se tapen.

In [ ]:
def perfil_robusto(s, por):
    g = s.groupby(por)
    return pd.DataFrame({'p25': g.quantile(0.25), 'p50': g.median(),
                         'p75': g.quantile(0.75), 'media': g.mean()})

obs_d = obs_diurna.dropna()
perf_h_orig = perfil_robusto(obs_d, obs_d.index.hour)
perf_m_orig = perfil_robusto(obs_d, obs_d.index.month)
perf_h_imp, perf_m_imp = {}, {}
for k in ['T1', 'T2', 'T3']:
    s = DATASETS[k].loc[MASK_DIURNO, 'RS']
    perf_h_imp[k] = perfil_robusto(s, s.index.hour)
    perf_m_imp[k] = perfil_robusto(s, s.index.month)

fig, axes = plt.subplots(2, 2, figsize=(14, 9))

# Fila 1: perfil horario. Izquierda con mediana y banda; derecha con media, para contraste.
ax = axes[0, 0]
ax.fill_between(perf_h_orig.index, perf_h_orig['p25'], perf_h_orig['p75'],
                color=C_GRIS_CLARO, alpha=0.55, label='Original, rango entre P25 y P75')
ax.plot(perf_h_orig.index, perf_h_orig['p50'], color=C_ORIG, lw=2.4, ls='--',
        label='Original, mediana')
for k in ['T1', 'T2', 'T3']:
    ax.plot(perf_h_imp[k].index, perf_h_imp[k]['p50'], color=COL[k], lw=1.8, label=ETQ[k])
ax.set_xlabel('Hora del dia'); ax.set_ylabel('Irradiancia (W/m$^2$)')
ax.set_xticks(range(6, 19)); ax.legend(frameon=False, fontsize=11); ax.grid(True, alpha=0.3)
ax.text(0.03, 0.96, 'Mediana y dispersion', transform=ax.transAxes, va='top', fontsize=12)

ax = axes[0, 1]
ax.plot(perf_h_orig.index, perf_h_orig['media'], color=C_ORIG, lw=2.4, ls='--',
        label='Original, promedio')
for k in ['T1', 'T2', 'T3']:
    ax.plot(perf_h_imp[k].index, perf_h_imp[k]['media'], color=COL[k], lw=1.8, label=ETQ[k])
ax.set_xlabel('Hora del dia'); ax.set_ylabel('Irradiancia (W/m$^2$)')
ax.set_xticks(range(6, 19)); ax.legend(frameon=False, fontsize=11); ax.grid(True, alpha=0.3)
ax.text(0.03, 0.96, 'Promedio, para contraste', transform=ax.transAxes, va='top', fontsize=12)

# Fila 2: perfil mensual, igual esquema
MESES2 = ['Ene','Feb','Mar','Abr','May','Jun','Jul','Ago','Sep','Oct','Nov','Dic']
ax = axes[1, 0]
ax.fill_between(perf_m_orig.index, perf_m_orig['p25'], perf_m_orig['p75'],
                color=C_GRIS_CLARO, alpha=0.55, label='Original, rango entre P25 y P75')
ax.plot(perf_m_orig.index, perf_m_orig['p50'], color=C_ORIG, lw=2.4, ls='--',
        label='Original, mediana')
for k in ['T1', 'T2', 'T3']:
    ax.plot(perf_m_imp[k].index, perf_m_imp[k]['p50'], color=COL[k], lw=1.8, label=ETQ[k])
ax.set_xticks(range(1, 13)); ax.set_xticklabels(MESES2, rotation=45)
ax.set_xlabel('Mes'); ax.set_ylabel('Irradiancia (W/m$^2$)')
ax.legend(frameon=False, fontsize=11); ax.grid(True, alpha=0.3)

ax = axes[1, 1]
ax.plot(perf_m_orig.index, perf_m_orig['media'], color=C_ORIG, lw=2.4, ls='--',
        label='Original, promedio')
for k in ['T1', 'T2', 'T3']:
    ax.plot(perf_m_imp[k].index, perf_m_imp[k]['media'], color=COL[k], lw=1.8, label=ETQ[k])
ax.set_xticks(range(1, 13)); ax.set_xticklabels(MESES2, rotation=45)
ax.set_xlabel('Mes'); ax.set_ylabel('Irradiancia (W/m$^2$)')
ax.legend(frameon=False, fontsize=11); ax.grid(True, alpha=0.3)

fig.tight_layout()
guardar_figura(fig, 'fig_09h_perfiles_robustos')
plt.show()

print('SEPARACION ENTRE ORIGINAL E IMPUTADAS, POR ESTADISTICA')
print('=' * 72)
for etq_p, po, pi in [('Perfil horario', perf_h_orig, perf_h_imp),
                      ('Perfil mensual', perf_m_orig, perf_m_imp)]:
    print(f'  {etq_p}')
    for k in ['T1', 'T2', 'T3']:
        d_med = (pi[k]['p50'] - po['p50']).abs().mean()
        d_avg = (pi[k]['media'] - po['media']).abs().mean()
        print(f'    {ETQ[k]:<26} mediana {d_med:7.2f}   promedio {d_avg:7.2f} W/m2')
print()
print('  LECTURA (corregida). El PROMEDIO practicamente coincide: los valores con que se')
print('  rellenan los huecos se toman de la climatologia observada, cuya media es la de')
print('  lo observado, de modo que el promedio se conserva casi por construccion.')
print('  La MEDIANA si se mueve, y bastante. La razon es que los valores imputados no')
print('  traen dispersion propia: se amontonan en el centro de la distribucion y no')
print('  reproducen ni los dias muy nublados ni los de realce. Al anadirlos, la mediana')
print('  sube y las colas se encogen.')
print()
print('  Eso se comprueba en la tabla descriptiva de la franja diurna:')
try:
    _o, _t = tabla_diurna['Original'], tabla_diurna['T1']
    print(f'    Desviacion estandar : {_o["Desv_est"]:7.2f}  ->  {_t["Desv_est"]:7.2f} W/m2')
    print(f'    Percentil 90        : {_o["P90"]:7.2f}  ->  {_t["P90"]:7.2f} W/m2')
    print(f'    Mediana             : {_o["Mediana"]:7.2f}  ->  {_t["Mediana"]:7.2f} W/m2')
except Exception:
    pass
print()
print('  CONSECUENCIA, y hay que declararla: el conjunto imputado es MENOS variable que')
print('  la realidad en los puntos que se rellenaron. El modelo se entrena sobre una')
print('  serie cuya nubosidad esta subrepresentada justo donde faltaban datos.')


## 10. Índice de claridad corregido

La figura del índice de claridad tiene un artefacto que conviene corregir antes de que lo
encuentre el jurado. El índice se define como el cociente entre la irradiancia observada y la
de cielo despejado. Cerca del amanecer y del ocaso el denominador tiende a cero, y el código
asigna cero al índice cuando eso ocurre. Esos ceros forzados no significan «cielo cubierto»:
significan «no hay denominador». Al entrar en el histograma y en la clasificación del cielo,
inflan artificialmente la categoría de cielo cubierto.

Se recalcula exigiendo un mínimo al denominador y se comparan las dos clasificaciones.

In [ ]:
UMBRAL_CLR = 20.0    # W/m2 minimos de irradiancia de cielo despejado

d = DATASETS['T1']
if {'ALLSKY_SFC_SW_DWN','CLRSKY_SFC_SW_DWN'}.issubset(d.columns):
    clr = d.loc[MASK_DIURNO, 'CLRSKY_SFC_SW_DWN']
    allsky = d.loc[MASK_DIURNO, 'ALLSKY_SFC_SW_DWN']
    kt_viejo = d.loc[MASK_DIURNO, 'KT']
    valido = clr > UMBRAL_CLR
    # Mismo techo que el cuaderno 02 revisado: 1.5. El indice puede superar la
    # unidad por realce de nubes, y acotarlo mas abajo recortaria esos episodios.
    kt_nuevo = (allsky[valido] / clr[valido]).clip(0, 1.5)

    def clasifica(kt):
        return pd.Series({
            'Cubierto'            : (kt < 0.3).sum(),
            'Parcialmente nublado': ((kt >= 0.3) & (kt < 0.6)).sum(),
            'Despejado'           : (kt >= 0.6).sum(),
        })

    cv, cn = clasifica(kt_viejo), clasifica(kt_nuevo)
    comp_kt = pd.DataFrame({
        'Version actual'      : cv,
        'Pct actual'          : (cv / cv.sum() * 100).round(2),
        'Con umbral'          : cn,
        'Pct con umbral'      : (cn / cn.sum() * 100).round(2),
    })
    print('CLASIFICACION DEL ESTADO DEL CIELO')
    print('=' * 70)
    print(comp_kt.to_string())
    print()
    print(f'  Registros descartados por denominador insuficiente: '
          f'{int((~valido).sum()):,} de {len(clr):,} ({(~valido).mean()*100:.2f} %)')
    print(f'  Media del indice, version actual : {kt_viejo.mean():.4f}')
    print(f'  Media del indice, con umbral     : {kt_nuevo.mean():.4f}')
    comp_kt.to_csv(os.path.join(RUTA_DATOS, 'tabla_09_kt_corregido.csv'))

    fig, axes = plt.subplots(1, 2, figsize=(14, 4.6))
    axes[0].hist(kt_viejo, bins=60, color=C_GRIS_CLARO, edgecolor='white', density=True,
                 label='Version actual')
    axes[0].hist(kt_nuevo, bins=60, color=C_T2, alpha=0.75, edgecolor='white',
                 density=True, label='Con umbral en el denominador')
    axes[0].axvline(0.3, color=C_ORIG, lw=1.2, ls='--')
    axes[0].axvline(0.6, color=C_ORIG, lw=1.2, ls='--')
    axes[0].set_xlabel('Indice de claridad')
    axes[0].set_ylabel('Densidad')
    axes[0].legend(frameon=False); axes[0].grid(True, alpha=0.3)

    kt_mes = kt_nuevo.resample('MS').mean()
    axes[1].plot(kt_mes.index, kt_mes.values, color=C_T2, lw=1.4)
    axes[1].set_xlabel('Fecha'); axes[1].set_ylabel('Indice de claridad promedio mensual')
    axes[1].grid(True, alpha=0.3)
    fig.tight_layout()
    guardar_figura(fig, 'fig_09g_kt_corregido')
    plt.show()
else:
    print('Las variables satelitales no estan en el conjunto. Omite esta seccion.')


## 11. Cuerpos de tabla listos para LaTeX

In [ ]:
def fmt(x, dec=2):
    if pd.isna(x):
        return '---'
    return f'{x:,.{dec}f}'.replace(',', '{,}')

FILAS = [('Numero de registros','N',0), ('Registros iguales a cero','Ceros',0),
         ('Minimo','Minimo',2), ('Maximo','Maximo',2), ('Media','Media',2),
         ('Mediana','Mediana',2), ('Desviacion estandar','Desv_est',2),
         ('CV (\\%)','CV_pct',2), ('Rango intercuartilico','RIC',2),
         ('P10','P10',2), ('P25','P25',2), ('P75','P75',2),
         ('P90','P90',2), ('P95','P95',2), ('P99','P99',2)]

for etq_t, T in [('FRANJA DIURNA', tabla_diurna), ('DIURNA POSITIVA', tabla_diurna_pos)]:
    print(f'% --- Cuerpo de la tabla comparativa, {etq_t} ---')
    for nom, key, dec in FILAS:
        vals = ' & '.join(fmt(T.loc[key, c], dec) for c in ['Original','T1','T2','T3'])
        print(f'{nom} & {vals} \\\\')
    print()

print('% --- Cuerpo de la tabla de validacion por ocultamiento ---')
for _, f in val_imp.iterrows():
    print(f"{f['tecnica']} & {f['clase']} & {fmt(f['n'],0)} & {fmt(f['MAE'])} & "
          f"{fmt(f['RMSE'])} & {fmt(f['sesgo'])} & {f['corr']:.3f} \\\\")


## 12. Qué llevar al documento

| Figura generada | Reemplaza o añade |
|---|---|
| `fig_09a_acf_pacf.pdf` | **Nueva.** Etapa 3 de resultados, análisis de series de tiempo |
| `fig_09b_descomposicion.pdf` | **Nueva.** Etapa 3, tendencia y estacionalidad |
| `fig_09c_multiescala.pdf` | **Reemplaza** la Figura 6.2 actual |
| `fig_09d_imputacion_de_cerca.pdf` | **Nueva.** Etapa 2, efecto de cada técnica |
| `fig_09e_validacion_imputacion.pdf` | **Nueva.** Etapa 2, evidencia de acierto |
| `fig_09f_conservacion_tendencia.pdf` | **Reemplaza** la Figura 6.3 actual |
| `fig_09g_kt_corregido.pdf` | **Reemplaza** la Figura 6.6 actual |

Los textos que acompañan a cada una están en `C1_metodologia_etapa2_y_3.tex` y
`C2_resultados_etapa2_y_3.tex`.